# STAGE 1

In this stage I build a tokenizer from scratch for data preparation and sampling purposes.

Here for dataset I am using WikiText-2 from huggingface and TinyStories.

Steps followed usually :     
1. Split the text into individual word and subwords
2. Convert these tokens into token IDs.

In [4]:
import time
import heapq

In [ ]:
class Trainer():
  def __init__(self, vocab_freq, num_merges):
    self.num_merges = num_merges
    self.merges = []

    self.vocab_freq = {}
    for word, freq in vocab_freq.items():
       tokens = tuple(word.split())
       self.vocab_freq[tokens] = freq

    self.pair_freq = {}
    for tokens, freq in self.vocab_freq.items():
       for i in range(len(tokens)-1):
          pair = (tokens[i], tokens[i+1])
          self.pair_freq[pair] = self.pair_freq.get(pair, 0 ) + freq
      
    self.heap = [(-freq, pair) for pair, freq in self.pair_freq.items()]
    heapq.heapify(self.heap)

  def get_best_pair(self):
    while self.heap:
        neg_freq, pair = heapq.heappop(self.heap)
        freq = -neg_freq

        if pair in self.pair_freq and self.pair_freq[pair] == freq:
            return pair

    return None

  def merge_vocab(self, word_pair):
      replacement = "".join(word_pair)
      new_vocab = {}
      for tokens, freq in self.vocab_freq.items():
            item = 0
            new_tokens = []
            while item < len(tokens) - 1 :
                if (tokens[item], tokens[item + 1]) == word_pair:
                    new_tokens.append(replacement)

                    if item > 0:
                       prev_pair = (tokens[item-1], tokens[item])
                       if prev_pair in self.pair_freq:
                            self.pair_freq[prev_pair] -= freq                    
                            if self.pair_freq[prev_pair] <= 0:  # <-- cleanup
                              del self.pair_freq[prev_pair]

                    if word_pair in self.pair_freq:  # <-- added check
                        self.pair_freq[word_pair] -= freq
                        if self.pair_freq[word_pair] <= 0:
                            del self.pair_freq[word_pair]

                    if item + 2 < len(tokens):
                      next_pair = (tokens[item+1], tokens[item+2])
                      if next_pair in self.pair_freq:  # <-- added check
                          self.pair_freq[next_pair] -= freq
                          if self.pair_freq[next_pair] <= 0:
                              del self.pair_freq[next_pair]


                    if item > 0:
                      new_prev = (tokens[item-1], replacement)
                      self.pair_freq[new_prev] = self.pair_freq.get(new_prev, 0) + freq

                    if item + 2 < len(tokens):
                        new_next = (replacement, tokens[item+2])
                        self.pair_freq[new_next] = self.pair_freq.get(new_next, 0) + freq
                    
                    item+=2

                else:
                    new_tokens.append(tokens[item])
                    item += 1

            if item == len(tokens) - 1:
                new_tokens.append(tokens[item])
            new_vocab[tuple(new_tokens)] = new_vocab.get(tuple(new_tokens), 0) + freq
      
      self.pair_freq = {k: v for k, v in self.pair_freq.items() if v >0}
      self.vocab_freq = new_vocab
      self.heap = [(-freq, pair) for pair, freq in self.pair_freq.items()]
      


  def loop(self):
      start_time = time.perf_counter()
      merge_rules= []

      for i in range(self.num_merges):
        best_pair = self.get_best_pair()

        if best_pair is None:
          break

        new_token = "".join(best_pair)
        merge_rules.append((best_pair,new_token))
        
        self.merge_vocab(best_pair)

        end_time = time.perf_counter()

        if ((i+1)%100==0):
          print(f"Iteration Number : {i+1}: Merge rules : {merge_rules}\n")
          print(f"Time used: {end_time - start_time:.6f} seconds")
          
      self.merges = merge_rules
      return merge_rules

In [24]:
vocab_freq_dummy_1 = vocab_freq

In [1]:
print(f"Trainer native timing for 100 iteration : 104.5 seconds")

Trainer native timing for 100 iteration : 104.5 seconds


In [2]:
print(f"Trainerv1 benchmark timing for 100 iterations : 53.64 seconds\n")

Trainerv1 benchmark timing for 100 iterations : 53.64 seconds



In [1]:
print(f"Trainerv1 benchmark timing for 9500 iterations : 1 hr 13 min\n")

Trainerv1 benchmark timing for 9500 iterations : 1 hr 13 min



In [ ]:
class 